# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulRaffayQureshi/FlyRank-ML-Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Our Lane: Lane 2 — Refresh / Content Optimization

- **Unit of Analysis:** One unique web page URL (`content_id` / `url_id`) tracked within a specific client domain context (`client_id`).
- **Aggregated Time Window:** A rolling 30-day window (with features reflecting the last 30 days and the historical baseline baseline window representing the previous period).
- **Temporal Grain Proof:** In our grain verification query below, we prove that the combination of `client_id` and `content_id` (or `url_id`) is completely unique per row within our target snapshot, with zero duplicated pairs. This ensures our unit of analysis matches our mathematical modeling frame.

In [1]:
import os
import pandas as pd
import numpy as np

# Ensure we can find the data from any directory structure in our repo
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    # Fallback in case we are running inside the notebooks directory
    data_path = "../../data/raw/content_refresh_anonymized.csv"
    if not os.path.exists(data_path):
        data_path = "../data/raw/content_refresh_anonymized.csv"

# Load local anonymized dataset
df = pd.read_csv(data_path)
print(f"Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.\n")

# Verify the temporal grain (Unique URL per Client)
duplicates = df.duplicated(subset=['client_id', 'content_id']).sum()
print(f"Verification 1 (The Grain):")
print(f"-> Total duplicated client-content rows: {duplicates}")
assert duplicates == 0, "Warning: The grain is not unique!"
print("-> SUCCESS: Each row is a guaranteed unique page per client domain.\n")

# Verify target date spans (if present)
print(f"Verification 2 (Time Window):")
if 'month' in df.columns:
    print(f"-> Months represented in data: {df['month'].unique()}")
else:
    print("-> Continuous snapshot mode: Data represents a single 30-day evaluation state across the 30k baseline tracking window.")

Dataset successfully loaded! Shape: 30000 rows, 44 columns.

Verification 1 (The Grain):
-> Total duplicated client-content rows: 0
-> SUCCESS: Each row is a guaranteed unique page per client domain.

Verification 2 (Time Window):
-> Continuous snapshot mode: Data represents a single 30-day evaluation state across the 30k baseline tracking window.


To maintain strict data boundaries and satisfy the **ML-04 Data Contract**, we organize our schema into four independent, functional buckets:

| Category | Fields Included | Justification / Engineering Boundaries |
| :--- | :--- | :--- |
| **1. Ingested Features** | `impressions_90d` (or `impressions_last_30d`), `avg_position`, `ctr`, `days_since_last_update`, `word_count` | Observable historical performance metrics that are fully knowable at the moment the editorial team decides to update a page. |
| **2. Target Label (Proxy)**| `is_declining_label` (derived from `trend_direction == 'down'`) | The ground-truth proxy of traffic decay. Utilized strictly for validation loss, metric calculation, and training loss optimization. |
| **3. Context Fields** | `client_id`, `content_id` | Restrictively used for database joins, grouping during custom splits (no client leakage), and sorting our final dashboard queue. *Completely omitted from the model’s features.* |
| **4. Excluded Fields** | `trend_direction`, `trend_pct` | **Deliberately Excluded:** These are directly derived from the target evaluation window. Ingesting them as features would trigger immediate, catastrophic target leakage. |

In [2]:
# Programmatically declare our Data Contract fields
MODEL_FEATURES = [
    "impressions_90d", 
    "days_since_last_update", 
    "avg_position", 
    "ctr", 
    "word_count"
]

EXCLUDED_LEAKAGE_FIELDS = ["trend_direction", "trend_pct"]

# Confirm that our feature space is target-blind and leakage-free
intersecting_leaks = set(MODEL_FEATURES).intersection(set(EXCLUDED_LEAKAGE_FIELDS))
print(f"Verification Check:")
print(f"-> Overlap between features and leak-prone target fields: {list(intersecting_leaks)}")
assert len(intersecting_leaks) == 0, "LEAK DETECTED: You are passing trend variables into features!"
print("-> SUCCESS: Ingested feature space is fully isolated from outcome-derived fields.")

Verification Check:
-> Overlap between features and leak-prone target fields: []
-> SUCCESS: Ingested feature space is fully isolated from outcome-derived fields.


In this section, we build our initial **5-Feature Frame** and explicitly trigger the **Target Leakage Trap**. 
We will train a simple decision boundary on a normal feature space, then intentionally inject `trend_pct` (our target-derived column) to show how our metric artificially jumps to near-perfection, before deleting it to preserve our honest baseline.

Every feature below is justified under the rule: **"Knowable at the decision moment."**
1. `impressions_90d`: Knowable because it represents real, historically recorded exposure before the decision point.
2. `days_since_last_update`: Knowable because the publishing CMS records this timestamp when content is modified.
3. `avg_position`: Knowable because search engines log daily positioning updates independently of future optimizations.
4. `ctr`: Knowable because click-through performance is calculated purely from past baseline exposure.
5. `word_count`: Knowable because the static page length is extracted directly from the existing page document.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# 1. Prepare target label and clean missing values
df_clean = df.copy()
df_clean["is_declining_label"] = df_clean["trend_direction"].str.lower().eq("down").astype(int)

# Fill missing values to keep pipeline robust
df_clean["days_since_last_update"] = df_clean["days_since_last_update"].fillna(180)
df_clean["word_count"] = df_clean["word_count"].fillna(df_clean["word_count"].median())

# 2. Build Honest 5-Feature Frame
X_honest = df_clean[MODEL_FEATURES]
y = df_clean["is_declining_label"]

# Train honest baseline model
clf_honest = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
clf_honest.fit(X_honest, y)
honest_preds = clf_honest.predict(X_honest)
honest_precision = precision_score(y, honest_preds, zero_division=0)

# 3. SPRING THE TRAP (Intentionally inject a target-derived leakage feature)
df_clean["trend_pct_leaked"] = df_clean["trend_pct"].fillna(0) # Leaked outcome field
X_leaked = df_clean[MODEL_FEATURES + ["trend_pct_leaked"]]

# Train on leaked data
clf_leaked = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
clf_leaked.fit(X_leaked, y)
leaked_preds = clf_leaked.predict(X_leaked)
leaked_precision = precision_score(y, leaked_preds, zero_division=0)

print("==== DATA CONTRACT LEAKAGE EXPERIMENT ====")
print(f"Honest Feature Frame Shape : {X_honest.shape}")
print(f"Honest Model Precision      : {honest_precision:.4f}")
print("-" * 42)
print(f"Leaked Feature Frame Shape : {X_leaked.shape}")
print(f"Leaked Model Precision      : {leaked_precision:.4f} (An artificial boost towards perfection!)")
print("==========================================")

# 4. Remove the leakage column programmatically to preserve the honest pipeline
del df_clean["trend_pct_leaked"]
assert "trend_pct_leaked" not in df_clean.columns
print("\n-> Leakage feature successfully deleted. The pipeline is safe and honest.")

==== DATA CONTRACT LEAKAGE EXPERIMENT ====
Honest Feature Frame Shape : (30000, 5)
Honest Model Precision      : 0.6048
------------------------------------------
Leaked Feature Frame Shape : (30000, 6)
Leaked Model Precision      : 1.0000 (An artificial boost towards perfection!)

-> Leakage feature successfully deleted. The pipeline is safe and honest.


### Named Limitations of Our Slice

While our dataset provides powerful signals, the data contract has explicit structural limitations that we must declare:

1. **The GSC Impressions-Only Threshold:** Our slice only captures metrics for pages that generated at least 1 impression in Google Search Console (GSC) over the past 90 days. Completely dead or index-blocked URLs (which have zero impressions) are entirely invisible in this slice. We cannot predict decay for pages that are not indexed.
2. **Missing Volatility Trends:** The rolling 30-day and 90-day averages flatten out acute daily traffic spikes. A sudden ranking crash occurring on Day 28 of the window will be heavily smoothed out by the preceding 27 days of high volume, introducing a minor detection lag in our decision-support queue.
3. **Lack of Off-Page Context:** The data contains no information regarding external ranking signals (such as backlink profile updates or competitor domain changes). If a page drops in ranking because a major competitor built 1,000 new backlinks, the model can only observe the *symptom* (decaying impressions), not the *cause*.

In [4]:
# Prove Limitation 1: Zero-impression pages are excluded from our historical slice
zero_impression_rows = (df["impressions_90d"] == 0).sum()
print(f"Verification of Data Limits:")
print(f"-> Number of rows with exactly 0 historical impressions: {zero_impression_rows}")
print("-> Analysis: This confirms that the slice is pre-filtered for active URLs only.")

Verification of Data Limits:
-> Number of rows with exactly 0 historical impressions: 0
-> Analysis: This confirms that the slice is pre-filtered for active URLs only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.